<a href="https://colab.research.google.com/github/etcex2969-spec/-AIFFEL_quest_eng/blob/main/%ED%8A%B8%EB%9E%9C%EC%8A%A4%ED%8F%AC%EB%A8%B8%20%EB%B2%88%EC%97%AD%EA%B8%B0%20%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


설명
Transformer 번역기  만들기.ipynb의 사본
Transformer 번역기  만들기.ipynb의 사본_

[ ]
!mkdir -p ~work/Transformer

[ ]
슝=3
내부 모듈 구현하기


[ ]
PyTorch version: 2.10.0+cu128
모든 라이브러리 임포트 완료!

[ ]

def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

print("슝=3")
슝=3

[ ]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // self.num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask):
        d_k = K.size(-1)
        QK = torch.matmul(Q, K.transpose(-2, -1))
        scaled_qk = QK / (d_k ** 0.5)
        if mask is not None:
            scaled_qk += (mask * -1e9)
        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)
        return out, attentions

    def split_heads(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def combine_heads(self, x):
        batch_size = x.size(0)
        return x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)

    def forward(self, Q, K, V, mask=None):
        WQ, WK, WV = self.W_q(Q), self.W_k(K), self.W_v(V)
        WQ_splits, WK_splits, WV_splits = self.split_heads(WQ), self.split_heads(WK), self.split_heads(WV)
        out, attention_weights = self.scaled_dot_product_attention(WQ_splits, WK_splits, WV_splits, mask)
        out = self.combine_heads(out)
        return self.linear(out), attention_weights

print("MultiHeadAttention defined.")
MultiHeadAttention defined.

[ ]
# MultiHeadAttention 테스트
# 하이퍼파라미터 설정
d_model = 512
num_heads = 8

# 모듈 초기화
mha = MultiHeadAttention(d_model, num_heads)

# 가상의 입력 데이터 생성 (batch_size=2, seq_len=10, d_model=512)
batch_size = 2
seq_len = 10
test_input = torch.randn(batch_size, seq_len, d_model)

# Forward Pass 수행
out, attn_weights = mha(test_input, test_input, test_input, mask=None)

print(f"입력 형태: {test_input.shape}")
print(f"출력 형태: {out.shape} (입력과 동일해야 함)")
print(f"어텐션 가중치 형태: {attn_weights.shape} (batch, heads, seq_len, seq_len)")

# 출력 검증
assert out.shape == (batch_size, seq_len, d_model)
print("\n슝=3 성공적으로 작동합니다!")
입력 형태: torch.Size([2, 10, 512])
출력 형태: torch.Size([2, 10, 512]) (입력과 동일해야 함)
어텐션 가중치 형태: torch.Size([2, 8, 10, 10]) (batch, heads, seq_len, seq_len)

슝=3 성공적으로 작동합니다!

[ ]
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.w_2(self.relu(self.w_1(x)))

print("PoswiseFeedForwardNet defined.")
PoswiseFeedForwardNet defined.
                  <모듈조립하기>
인코더 레이어 구현하기


[ ]
import torch
import torch.nn as nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn

print("슝=3")
슝=3
디코더 레이어 구현하기


[ ]
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        # 1. Masked Multi-Head Attention (Self-Attention)
        # 디코더의 현재 시점까지만 보도록 causality_mask를 사용합니다.
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, causality_mask)
        out = self.dropout(out)
        out += residual

        # 2. Multi-Head Attention (Encoder-Decoder Attention)
        # 인코더의 출력(K, V)을 보되, 인코더의 패딩을 가리기 위해 padding_mask를 사용합니다.
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, padding_mask)
        out = self.dropout(out)
        out += residual

        # 3. Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn

print("DecoderLayer redefined correctly.")
DecoderLayer redefined correctly.
Transformer


[ ]
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, src_vocab_size, tgt_vocab_size, pos_len, dropout=0.2, shared=True):
        super(Transformer, self).__init__()
        self.d_model = float(d_model)

        # 1. 임베딩(Embedding) 레이어: 단어 번호를 연속적인 벡터 공간으로 변환
        self.enc_emb = nn.Embedding(src_vocab_size, d_model) # 소스(입력) 언어용
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model) # 타겟(출력) 언어용

        # 2. 포지셔널 인코딩: 단어의 위치 정보를 수치화해서 저장
        self.pos_encoding = self.positional_encoding(pos_len, d_model)
        self.dropout = nn.Dropout(dropout)

        # 3. 인코더 & 디코더: 트랜스포머의 핵심 엔진 조립 (여러 층 쌓기)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        # 4. 출력 레이어: 벡터를 다시 단어별 확률(Logits)로 변환
        self.fc = nn.Linear(d_model, tgt_vocab_size)

        # 5. 가중치 공유(Weight Sharing): 출력층과 디코더 임베딩 가중치를 통일 (학습 효율 상승)
        self.shared = shared
        if shared:
            self.fc.weight = self.dec_emb.weight

    def positional_encoding(self, pos_len, d_model):
        """
        단어의 상대적 위치를 나타내는 사인/코사인 파동 그래프를 생성
        """
        position = torch.arange(0, pos_len).unsqueeze(1).float()
        # 주기함수의 주기를 결정하는 분모 항 계산
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))

        pos_encoding = torch.zeros(pos_len, d_model)
        pos_encoding[:, 0::2] = torch.sin(position * div_term) # 짝수 인덱스: 사인 함수
        pos_encoding[:, 1::2] = torch.cos(position * div_term) # 홀수 인덱스: 코사인 함수
        return pos_encoding

    def embedding(self, emb, x):
        """
        단어를 벡터로 바꾸고 + 위치 정보(Positional Encoding)를 더해주는 과정
        """
        seq_len = x.size(1)
        out = emb(x) # 단어 -> 벡터

        if self.shared:
            # 가중치를 공유할 경우, 스케일을 조정해서 임베딩 값을 키워줌
            out *= torch.sqrt(torch.tensor(self.d_model))

        # 미리 계산한 위치 정보(pos_encoding)를 단어 벡터에 더함
        out += self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.dropout(out)

        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        """
        트랜스포머 엔진 가동! (데이터가 흐르는 전체 경로)
        """
        # 1. 입력 데이터를 임베딩 + 위치 정보와 결합
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        # 2. 인코더 통과: 입력 문장의 맥락을 파악 (Self-Attention)
        enc_out, enc_attns = self.encoder(enc_in, enc_mask)

        # 3. 디코더 통과: 인코더의 정보와 이전 단어들을 보고 다음 단어 예측
        dec_out, dec_attns, dec_enc_attns = \
            self.decoder(dec_in, enc_out, causality_mask, dec_mask)

        # 4. 최종 결과물(Logits) 생성
        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns

[ ]

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, src_vocab_size, tgt_vocab_size, pos_len, dropout=0.2, shared=True):
        super(Transformer, self).__init__()
        self.d_model = float(d_model)

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        self.pos_encoding = self.positional_encoding(pos_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared = shared
        if shared:
            self.fc.weight = self.dec_emb.weight

    def positional_encoding(self, pos_len, d_model):
        position = torch.arange(0, pos_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pos_encoding = torch.zeros(pos_len, d_model)
        pos_encoding[:, 0::2] = torch.sin(position * div_term)
        pos_encoding[:, 1::2] = torch.cos(position * div_term)
        return pos_encoding

    def embedding(self, emb, x):
        seq_len = x.size(1)
        out = emb(x)
        if self.shared:
            out *= torch.sqrt(torch.tensor(self.d_model))
        device = out.device
        out += self.pos_encoding[:seq_len, :].to(device).unsqueeze(0)
        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)
        # 디코더의 forward 인자 순서에 맞춰 전달: (x, enc_out, causality_mask, padding_mask)
        # dec_mask는 causality와 padding이 합쳐진 것, dec_enc_mask는 인코더 패딩 가리기용입니다.
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, dec_mask, dec_enc_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

[ ]
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask):
        attns = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            attns.append(attn)
        return self.norm(x), attns

[ ]
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        self_attns, dec_enc_attns = [], []
        for layer in self.layers:
            x, self_attn, dec_enc_attn = layer(x, enc_out, causality_mask, padding_mask)
            self_attns.append(self_attn)
            dec_enc_attns.append(dec_enc_attn)
        return self.norm(x), self_attns, dec_enc_attns

[ ]
# Transformer 최종 테스트
# 하이퍼파라미터 설정
n_layers = 2
d_model = 512
n_heads = 8
d_ff = 2048
src_vocab_size = 100
tgt_vocab_size = 100
pos_len = 50
dropout = 0.1

# 모델 생성
transformer = Transformer(
    n_layers, d_model, n_heads, d_ff,
    src_vocab_size, tgt_vocab_size, pos_len, dropout
)

# 더미 데이터 생성 (batch_size=2, seq_len=10)
batch_size = 2
seq_len = 10
src_input = torch.randint(0, src_vocab_size, (batch_size, seq_len))
tgt_input = torch.randint(0, tgt_vocab_size, (batch_size, seq_len))

# 마스크 생성 (더미)
enc_mask = torch.zeros(batch_size, 1, 1, seq_len)
causality_mask = torch.zeros(batch_size, 1, seq_len, seq_len)
dec_mask = torch.zeros(batch_size, 1, 1, seq_len)

# Forward Pass
logits, enc_attns, dec_attns, dec_enc_attns = transformer(
    src_input, tgt_input, enc_mask, causality_mask, dec_mask
)

print(f"★ ★ ★ Transformer Test Results ★ ★ ★")
print(f"Logits Shape: {logits.shape} (Expected: [2, 10, 100])")
print(f"Encoder Attention Layers: {len(enc_attns)}")
print(f"Decoder Attention Layers: {len(dec_attns)}")

assert logits.shape == (batch_size, seq_len, tgt_vocab_size)
print("\n슝=3 슝=3 슝=3 슝=3 슝=3 성공적으로 작동합니다!")
★ ★ ★ Transformer Test Results ★ ★ ★
Logits Shape: torch.Size([2, 10, 100]) (Expected: [2, 10, 100])
Encoder Attention Layers: 2
Decoder Attention Layers: 2

슝=3 슝=3 슝=3 슝=3 슝=3 성공적으로 작동합니다!

[ ]
# 손실 함수(Loss Function) 및 옵티마이저(Optimizer) 설정

# 1. CrossEntropyLoss: 다중 클래스 분류를 위한 손실 함수
# ignore_index=0: 패딩 토큰(0)에 대한 손실은 계산하지 않도록 설정
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 2. Optimizer: Adam 옵티마이저 사용
learning_rate = 0.0001
optimizer = optim.Adam(transformer.parameters(), lr=learning_rate)

print("손실 함수와 옵티마이저 설정 완료!")
print(f"설정된 Learning Rate: {learning_rate}")
손실 함수와 옵티마이저 설정 완료!
설정된 Learning Rate: 0.0001
📝 Transformer 학습 과정 요약
데이터 전처리 및 임베딩:

소스(Source) 문장과 타겟(Target) 문장을 술자 토큰으룜 변환합니다.
단어 벡터에 위치 정보림 더하는 Positional Encoding을 실행합니다.
모델 통과 (Forward Pass):

Encoder: 입력 문장의 문맥을 파악하여 벡터룜 압축합니다.
Decoder: 이전에 예측한 단어들과 인코더의 출력을 참고하여 다음 단어림 예측합니다. 이때 미래의 단어림 보지 못하도록 Look-ahead Mask림 적용합니다.
손실 계산 (Loss Calculation):

CrossEntropyLoss: 모델이 예측한 단어 확률 분포와 실제 정답 단어림 비교합니다.
ignore_index=0: 문장 길을 맞추기 위해 채운 패딩() 토큰은 학습에 영향을 준지 않도록 손실 계산에서 제외합니다.
가중치 업녰이트 (Optimization):

Adam Optimizer: 계산된 손실값을 바탕으룜 모델의 파트너미터림 조절합니다. 효율적인 협습을 위해 기울기(Gradient)림 업녰이트합니다.
반복 (Iteration):

위 과정을 수많은 데이터에 대해 반복(에포크, Epoch)하여 모델의 정‐도림 높입니다.

[ ]
def evaluate(model, data_loader, criterion, device):
    model.eval() # 평가 모드 전환 (드롭아웃 비활성화)
    total_loss = 0

    with torch.no_grad(): # 그래디언트 계산 비활성화 (메모리 절약)
        for batch in data_loader:
            # 데이터를 장치(CPU/GPU)로 이동
            src_input, tgt_input, tgt_output, enc_mask, causality_mask, dec_mask = [b.to(device) for b in batch]

            # Forward Pass
            logits, _, _, _ = model(
                src_input, tgt_input, enc_mask, causality_mask, dec_mask
            )

            # Loss 계산
            # logits: [batch, seq_len, vocab_size] -> [batch * seq_len, vocab_size]
            # tgt_output: [batch, seq_len] -> [batch * seq_len]
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_output.view(-1))
            total_loss += loss.item()

    return total_loss / len(data_loader)

print("평가(Evaluation) 함수 정의 완료!")
평가(Evaluation) 함수 정의 완료!

[ ]
def train(model, data_loader, optimizer, criterion, device):
    model.train() # 학습 모드 전환
    total_loss = 0

    for i, batch in enumerate(data_loader):
        # 데이터를 장치로 이동
        src_input, tgt_input, tgt_output, enc_mask, causality_mask, dec_mask = [b.to(device) for b in batch]

        # 1. Gradient 초기화
        optimizer.zero_grad()

        # 2. Forward Pass
        logits, _, _, _ = model(
            src_input, tgt_input, enc_mask, causality_mask, dec_mask
        )

        # 3. Loss 계산
        loss = criterion(logits.view(-1, logits.size(-1)), tgt_output.view(-1))

        # 4. Backward Pass (기울기 계산)
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        total_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(f"  Batch {i+1} Loss: {loss.item():.4f}")

    return total_loss / len(data_loader)

# --- 실제 학습 실행 예시 (가상 데이터 로더 필요) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer.to(device)

EPOCHS = 10
print(f"{device} 장치에서 학습을 시작합니다.")

# for epoch in range(EPOCHS):
#     train_loss = train(transformer, train_loader, optimizer, criterion, device)
#     val_loss = evaluate(transformer, val_loader, criterion, device)
#     print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("학습 루프 함수 정의 완료!")
cuda 장치에서 학습을 시작합니다.
학습 루프 함수 정의 완료!
모델밖의 조력자들(상세주석)


[ ]
def generate_padding_mask(seq):
    mask = (seq == 0).float()
    # [batch, 1, 1, seq_len] 형태로 반환하여 브로드캐스팅 지원
    return mask.unsqueeze(1).unsqueeze(2)

def generate_causality_mask(size):
    # 미래 단어를 가리는 상삼각 행렬
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def generate_masks(src, tgt):
    # 1. 인코더 셀프 어텐션용 패딩 마스크
    enc_mask = generate_padding_mask(src)

    # 2. 디코더-인코더 어텐션용 패딩 마스크 (인코더의 패딩을 가림)
    # 디코더가 인코더의 K, V를 보므로 인코더(src)의 패딩 정보를 가려야 합니다.
    dec_enc_mask = generate_padding_mask(src)

    # 3. 디코더 셀프 어텐션용 마스크 (미래 가리기 + 패딩 가리기)
    tgt_seq_len = tgt.shape[1]
    # 미래 가리기 (Causality)
    causal_mask = generate_causality_mask(tgt_seq_len).to(tgt.device)
    # 디코더 패딩 가리기
    dec_padding_mask = generate_padding_mask(tgt).to(tgt.device)

    # 두 마스크를 합쳐서 하나라도 1(가림)이면 가리도록 함
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("generate_masks function updated with dynamic sequence length support.")
generate_masks function updated with dynamic sequence length support.

[ ]
import matplotlib.pyplot as plt

# 1. 테스트용 가짜 데이터 설정
batch, length = 16, 20     # 배치 크기 16, 실제 데이터 길이 20
src_padding = 5            # 입력 문장에 붙일 패딩 길이 5
tgt_padding = 15           # 출력 문장에 붙일 패딩 길이 15

# 2. 패딩 데이터 생성 (0으로 채워진 텐서)
src_pad = torch.zeros((batch, src_padding))
tgt_pad = torch.zeros((batch, tgt_padding))

# 3. 실제 데이터 생성 (1로 채워진 텐서) 후 패딩과 합치기 (Concatenate)
sample_data = torch.ones((batch, length))
sample_src = torch.cat([sample_data, src_pad], dim=-1) # 최종 입력: [16, 25]
sample_tgt = torch.cat([sample_data, tgt_pad], dim=-1) # 최종 출력: [16, 35]

# 4. 아까 만든 함수로 마스크 3종 세트 생성
enc_mask, dec_enc_mask, dec_mask = generate_masks(sample_src, sample_tgt)

# 5. 시각화 준비 (한 화면에 그래프 3개 그리기)
fig = plt.figure(figsize=(7, 7))

ax1 = fig.add_subplot(131) # 인코더 셀프 어텐션용 마스크
ax2 = fig.add_subplot(132) # 디코더-인코더 어텐션용 마스크
ax3 = fig.add_subplot(133) # 디코더 셀프 어텐션용 (미래 가리기 + 패딩 가리기)

ax1.set_title('1) Encoder Mask')
ax2.set_title('2) Encoder-Decoder Mask')
ax3.set_title('3) Decoder Mask')

# 6. 마스크 이미지 출력 (Dark2 컬러맵 사용)
# enc_mask[:3, 0, 0]: 여러 배치 중 일부만 샘플로 확인
ax1.imshow(enc_mask[:3, 0, 0].numpy(), cmap='Dark2')
ax2.imshow(dec_enc_mask[0, 0].numpy(), cmap='Dark2')
ax3.imshow(dec_mask[0, 0].numpy(), cmap='Dark2')

plt.show() # "슝=3" 하고 그래프 출력!

Encoder Mask:

문장 뒤쪽의 5칸(Padding)이 색칠되어 보일 겁니다. "여기는 정보가 없으니 무시해!"라는 뜻이죠.

Encoder-Decoder Mask:

디코더가 인코더를 볼 때, 인코더 문장의 패딩 부분을 보지 못하게 막아줍니다.

Decoder Mask:

이게 하이라이트! 계단 모양(삼각형)의 대각선 가림막이 보일 거예요.

"현재 단어를 예측할 때 오른쪽(미래) 단어는 절대 보지 마!"라고 엄격하게 가리고 있는 모습입니다.


[ ]
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = torch.tensor(step, dtype=torch.float32)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * torch.minimum(arg1, arg2)

model = nn.Linear(10, 10)
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-9,
    betas=(0.9, 0.98),
    eps=1e-9)

print("슝=3")
슝=3
                                       (프로젝트)
                               더 멋진 번역기 만들기

[ ]
import torch
import numpy
import matplotlib

print(torch.__version__)
print(numpy.__version__)
print(matplotlib.__version__)
2.10.0+cu128
2.0.2
3.10.0

[ ]
!mkdir -p ~/work/transformer/data
%cd ~/work/transformer/data

!wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
!gzip -d korean-english-park.train.tar.gz
!tar -xvf korean-english-park.train.tar
/root/work/transformer/data
--2026-05-13 03:21:50--  https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz [following]
--2026-05-13 03:21:51--  https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8718893 (8.3M) [application/octet-stream]
Saving to: ‘korean-english-park.train.tar.gz’

korean-english-park 100%[===================>]   8.31M  --.-KB/s    in 0.06s

2026-05-13 03:21:51 (146 MB/s) - ‘korean-english-park.train.tar.gz’ saved [8718893/8718893]

korean-english-park.train.en
korean-english-park.train.ko

[ ]
!wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz -P ~/work/transformer/data
!gzip -d ~/work/transformer/data/korean-english-park.train.tar.gz
!tar -xvf ~/work/transformer/data/korean-english-park.train.tar
--2026-05-13 03:21:52--  https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz [following]
--2026-05-13 03:21:52--  https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8718893 (8.3M) [application/octet-stream]
Saving to: ‘/root/work/transformer/data/korean-english-park.train.tar.gz’

korean-english-park 100%[===================>]   8.31M  --.-KB/s    in 0.06s

2026-05-13 03:21:52 (143 MB/s) - ‘/root/work/transformer/data/korean-english-park.train.tar.gz’ saved [8718893/8718893]

gzip: /root/work/transformer/data/korean-english-park.train.tar already exists; do you wish to overwrite (y or n)?
                                  데이터 정제 및 토큰화

[ ]
data_dir = os.path.join(os.getenv("HOME"), 'work/transformer/data')
kor_path = data_dir+"/korean-english-park.train.ko"
eng_path = data_dir+"/korean-english-park.train.en"

# 데이터 정제 및 토큰화
def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r") as f: kor = f.read().splitlines()
    with open(eng_path, "r") as f: eng = f.read().splitlines()
    assert len(kor) == len(eng)

    cleaned_corpus = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))

    return cleaned_corpus

cleaned_corpus = clean_corpus(kor_path, eng_path)

[ ]
def preprocess_sentence(sentence):
    # 1. 소문자 변환: 모든 영문자를 소문자로 통일 (Apple과 apple을 같은 단어로 처리)
    sentence = sentence.lower()

    # 2. 노이즈 제거: 알파벳(a-z), 한글(가-힣), 주요 문장부호(?.!,)를 제외한
    # 모든 특수문자나 숫자 등을 공백(" ")으로 치환합니다.
    # [^...]는 '이 안에 없는 것들'을 의미해요!
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)

    # 3. 문장부호 분리: 문장부호(?, ., !, ,) 앞에 공백을 추가하여 단어와 분리합니다.
    # 예: "안녕?" -> "안녕 ?" (이렇게 해야 모델이 '안녕'과 '?'를 각각 학습할 수 있어요)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 4. 연속 공백 제거: 여러 개의 공백("  ")이 생겼을 경우 하나(" ")로 줄입니다.
    sentence = re.sub(r'[" "]+', " ", sentence)

    return sentence
단어 사전을 매개변수로 받아 원하는 크기의 사전을 정의할 수 있게 합니다. (기본: 20,000) 학습 후 저장된 model 파일을 SentencePieceProcessor() 클래스에 Load() 한 후 반환합니다. 특수 토큰의 인덱스를 아래와 동일하게 지정합니다. : 0 / : 1 / : 2 / : 3


[ ]
# Sentencepiece를 활용하여 학습한 tokenizer를 생성합니다.
def generate_tokenizer(corpus,
                        vocab_size,
                        lang="ko",
                        pad_id=0,
                        bos_id=1,
                        eos_id=2,
                        unk_id=3):
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
      for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
      '--input=./%s --model_prefix=%s --vocab_size=%d'\
      % (file, model, vocab_size) + \
      '--pad_id==%d --bos_id=%d --eos_id=%d --unk_id=%d'\
      % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer


SRC_VOCAB_SIZE = TGT_VOCAB_SIZE = 20000

eng_corpus = []
kor_corpus = []

for pair in cleaned_corpus:
    k, e = pair.split("\t")

    kor_corpus.append(preprocess_sentence(k))
    eng_corpus.append(preprocess_sentence(e))

ko_tokenizer = generate_tokenizer(kor_corpus, SRC_VOCAB_SIZE, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, TGT_VOCAB_SIZE, "en")
en_tokenizer.set_encode_extra_options("bos:eos")

[ ]
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm  # 진행 과정 보기

src_corpus = []
tgt_corpus = []

assert len(kor_corpus) == len(eng_corpus)

# 토큰의 길이가 50 이하인 문장만 남깁니다.
for idx in tqdm(range(len(kor_corpus))):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])

    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus.append(torch.tensor(tgt_tokens, dtype=torch.long))

def pad_sequences(sequences, padding_value=0):
    return torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=padding_value)

# 패딩처리를 완료하여 학습용 데이터를 완성합니다.
enc_train = pad_sequences(src_corpus, padding_value=0)
dec_train = pad_sequences(tgt_corpus, padding_value=0)

print(enc_train.shape, dec_train.shape)
모델설계


[ ]
N_LAYERS = 2
D_MODEL = 512
N_HEADS = 8
D_FF = 2048
DROPOUT = 0.1

transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=50,
    dropout=DROPOUT,
    shared=True
).to(device)

optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Transformer model re-instantiated with fixed mask routing.")

[ ]
from torch.utils.data import DataLoader, TensorDataset

# 1. 하이퍼파라미터 및 배치 설정
BATCH_SIZE = 64

# 2. Dataset 및 DataLoader 생성
# dec_train을 입력(tgt_input)과 출력(tgt_output)으로 분리
# tgt_input: <BOS> 토큰부터 마지막 전까지
# tgt_output: <BOS> 다음 토큰부터 <EOS>까지
train_dataset = TensorDataset(
    enc_train,
    dec_train[:, :-1],
    dec_train[:, 1:]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"DataLoader 설정 완료! 총 배치 수: {len(train_loader)}")

# 3. 데이터 샘플 확인
for src, tgt_in, tgt_out in train_loader:
    print(f"Source Batch Shape: {src.shape}")
    print(f"Target Input Shape: {tgt_in.shape}")
    print(f"Target Output Shape: {tgt_out.shape}")
    break

[ ]
from sklearn.model_selection import train_test_split

# 1. 데이터를 학습용과 검증용으로 분리 (8:2)
enc_train_split, enc_val_split, dec_train_split, dec_val_split = train_test_split(
    enc_train, dec_train, test_size=0.2, random_state=42
)

# 2. DataLoader 생성
# Train Loader
train_dataset = TensorDataset(
    enc_train_split,
    dec_train_split[:, :-1],
    dec_train_split[:, 1:]
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Validation Loader
val_dataset = TensorDataset(
    enc_val_split,
    dec_val_split[:, :-1],
    dec_val_split[:, 1:]
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"데이터 분리 완료!")
print(f"Train Data: {len(enc_train_split)} | Val Data: {len(enc_val_split)}")

[ ]
def evaluate_validation(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            src, tgt_in, tgt_out = [b.to(device) for b in batch]

            # 마스크 생성 및 장치 이동
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
            enc_mask = enc_mask.to(device)
            dec_enc_mask = dec_enc_mask.to(device)
            dec_mask = dec_mask.to(device)

            # Forward
            logits, _, _, _ = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

            # Loss
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            total_loss += loss.item()

    return total_loss / len(data_loader)

print("검증(evaluate_validation) 함수 이름 변경 완료!")

[ ]
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import re
import torch
import sentencepiece as spm

# 1. Utility function for preprocessing
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[\" \"]+', " ", sentence)
    return sentence

# 2. 데이터 디렉토리 설정 및 데이터 다운로드
data_dir = os.path.join(os.getenv("HOME"), 'work/Transformer/data')
kor_path = os.path.join(data_dir, "korean-english-park.train.ko")
eng_path = os.path.join(data_dir, "korean-english-park.train.en")

if not os.path.exists(kor_path):
    print("Data files not found. Downloading and extracting...")
    os.makedirs(data_dir, exist_ok=True)
    !wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz -P {data_dir}
    !tar -xvf {data_dir}/korean-english-park.train.tar.gz -C {data_dir}

# 3. 데이터 로드 및 정제
def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r", encoding='utf-8') as f: kor = f.read().splitlines()
    with open(eng_path, "r", encoding='utf-8') as f: eng = f.read().splitlines()
    cleaned = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))
    return cleaned

cleaned_corpus = clean_corpus(kor_path, eng_path)
kor_corpus = []
eng_corpus = []
for pair in cleaned_corpus:
    parts = pair.split("\t")
    if len(parts) == 2:
        k, e = parts
        kor_corpus.append(preprocess_sentence(k))
        eng_corpus.append(preprocess_sentence(e))

# 4. 토크나이저 생성
def generate_tokenizer(corpus, lang="ko", vocab_size=20000):
    file = f"./{lang}_corpus.txt"
    model_prefix = f"{lang}_spm"
    with open(file, 'w', encoding='utf-8') as f:
        for row in corpus: f.write(str(row) + '\n')
    spm.SentencePieceTrainer.Train(f'--input={file} --model_prefix={model_prefix} --vocab_size={vocab_size} --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3')
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model_prefix}.model')
    return tokenizer

ko_tokenizer = generate_tokenizer(kor_corpus, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, "en")
en_tokenizer.set_encode_extra_options("bos:eos")

# 5. 토큰화 및 패딩
src_corpus_ids = []
tgt_corpus_ids = []
for idx in range(len(kor_corpus)):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])
    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus_ids.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus_ids.append(torch.tensor(tgt_tokens, dtype=torch.long))

enc_train = torch.nn.utils.rnn.pad_sequence(src_corpus_ids, batch_first=True, padding_value=0)
dec_train = torch.nn.utils.rnn.pad_sequence(tgt_corpus_ids, batch_first=True, padding_value=0)

# 6. DataLoader 설정
BATCH_SIZE = 64
enc_train_split, enc_val_split, dec_train_split, dec_val_split = train_test_split(enc_train, dec_train, test_size=0.2, random_state=42)

train_dataset = TensorDataset(enc_train_split, dec_train_split[:, :-1], dec_train_split[:, 1:])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(enc_val_split, dec_val_split[:, :-1], dec_val_split[:, 1:])
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders initialized: {len(train_loader)} train batches.")

[ ]
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# --- 1. Transformer Architecture Definition ---
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)
    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)
    def split_heads(self, x):
        batch_size = x.size(0)
        return x.view(batch_size, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        q, k, v = self.W_q(Q), self.W_k(K), self.W_v(V)
        q, k, v = self.split_heads(q), self.split_heads(k), self.split_heads(v)
        scaled_qk = torch.matmul(q, k.transpose(-2, -1)) / (k.size(-1)**0.5)
        if mask is not None: scaled_qk += (mask * -1e9)
        attn = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attn, v).permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)
        return self.linear(out), attn

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
    def forward(self, x): return self.w_2(F.relu(self.w_1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1, self.norm2 = nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask):
        out, _ = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(out))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x, _

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, n_heads)
        self.mha2 = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6), nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, causality_mask, padding_mask):
        out, _ = self.mha1(x, x, x, causality_mask)
        x = self.norm1(x + self.dropout(out))
        out, _ = self.mha2(x, enc_out, enc_out, padding_mask)
        x = self.norm2(x + self.dropout(out))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, _, _

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, src_vocab_size, tgt_vocab_size, pos_len, dropout=0.2, shared=True):
        super().__init__()
        self.d_model = float(d_model)
        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = torch.tensor(positional_encoding(pos_len, d_model), dtype=torch.float32)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        if shared: self.fc.weight = self.dec_emb.weight
    def embedding(self, emb, x):
        out = emb(x) * (self.d_model**0.5) + self.pos_encoding[:x.size(1), :].to(x.device).unsqueeze(0)
        return self.dropout(out)
    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        x = self.embedding(self.enc_emb, enc_in)
        for layer in self.encoder: x, _ = layer(x, enc_mask)
        enc_out = x
        y = self.embedding(self.dec_emb, dec_in)
        for layer in self.decoder: y, _, _ = layer(y, enc_out, dec_mask, dec_enc_mask)
        return self.fc(y), None, None, None

# --- 2. Helper Functions ---
def generate_masks(src, tgt):
    enc_mask = (src == 0).float().unsqueeze(1).unsqueeze(2)
    dec_enc_mask = (src == 0).float().unsqueeze(1).unsqueeze(2)
    causal_mask = torch.triu(torch.ones(tgt.shape[1], tgt.shape[1]), diagonal=1).to(tgt.device)
    dec_padding_mask = (tgt == 0).float().unsqueeze(1).unsqueeze(2)
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)
    return enc_mask, dec_enc_mask, dec_mask

# --- 3. Execution ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure Model is defined
if 'transformer' not in globals():
    transformer = Transformer(n_layers=2, d_model=512, n_heads=8, d_ff=2048, src_vocab_size=20000, tgt_vocab_size=20000, pos_len=200).to(device)
    optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

# Ensure Data Loaders are defined
if 'train_loader' not in globals():
    if 'enc_train' in globals() and 'dec_train' in globals():
        enc_tr, enc_val, dec_tr, dec_val = train_test_split(enc_train, dec_train, test_size=0.2, random_state=42)
        train_loader = DataLoader(TensorDataset(enc_tr, dec_tr[:, :-1], dec_tr[:, 1:]), batch_size=64, shuffle=True)
        val_loader = DataLoader(TensorDataset(enc_val, dec_val[:, :-1], dec_val[:, 1:]), batch_size=64, shuffle=False)
    else:
        raise NameError("Missing 'enc_train' or 'dec_train' variables. Please run the preprocessing cells.")

EPOCHS = 5
for epoch in range(EPOCHS):
    transformer.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        src, tgt_in, tgt_out = [b.to(device) for b in batch]
        e_m, de_m, d_m = generate_masks(src, tgt_in)
        optimizer.zero_grad()
        logits, _, _, _ = transformer(src, tgt_in, e_m, de_m, d_m)
        loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
        loss.backward(); optimizer.step()
        total_loss += loss.item()

    transformer.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            src, tgt_in, tgt_out = [b.to(device) for b in batch]
            e_m, de_m, d_m = generate_masks(src, tgt_in)
            logits, _, _, _ = transformer(src, tgt_in, e_m, de_m, d_m)
            val_loss += criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1)).item()
    print(f"Avg Train Loss: {total_loss/len(train_loader):.4f} | Avg Val Loss: {val_loss/len(val_loader):.4f}")

[ ]
import os
import re
import torch
import sentencepiece as spm
from sklearn.model_selection import train_test_split

# 1. Utility: Preprocessing function
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    return sentence.strip()

# 2. Utility: Tokenizer Generator
def generate_tokenizer(corpus, lang="ko", vocab_size=20000):
    file = f"./{lang}_corpus.txt"
    model_prefix = f"{lang}_spm"
    with open(file, 'w', encoding='utf-8') as f:
        for row in corpus: f.write(str(row) + '\n')

    spm.SentencePieceTrainer.Train(
        f'--input={file} --model_prefix={model_prefix} --vocab_size={vocab_size} '
        f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3'
    )
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model_prefix}.model')
    return tokenizer

# 3. Data Loading and Cleaning
data_dir = os.path.join(os.getenv('HOME'), 'work/Transformer/data')
kor_path = os.path.join(data_dir, 'korean-english-park.train.ko')
eng_path = os.path.join(data_dir, 'korean-english-park.train.en')

if not os.path.exists(kor_path):
    print("Data not found, downloading...")
    os.makedirs(data_dir, exist_ok=True)
    !wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz -P {data_dir}
    !tar -xvf {data_dir}/korean-english-park.train.tar.gz -C {data_dir}

def clean_corpus(kor_path, eng_path):
    with open(kor_path, 'r', encoding='utf-8') as f: kor = f.read().splitlines()
    with open(eng_path, 'r', encoding='utf-8') as f: eng = f.read().splitlines()
    cleaned = list(set(['\t'.join([k, e]) for k, e in zip(kor, eng)]))
    return cleaned

print("Cleaning corpus...")
cleaned_corpus = clean_corpus(kor_path, eng_path)
kor_corpus = [preprocess_sentence(pair.split('\t')[0]) for pair in cleaned_corpus if '\t' in pair]
eng_corpus = [preprocess_sentence(pair.split('\t')[1]) for pair in cleaned_corpus if '\t' in pair]

# 4. Tokenization
print("Generating tokenizers...")
ko_tokenizer = generate_tokenizer(kor_corpus, lang='ko', vocab_size=20000)
en_tokenizer = generate_tokenizer(eng_corpus, lang='en', vocab_size=20000)
en_tokenizer.set_encode_extra_options('bos:eos')

# 5. Tensor Creation
print("Converting to tensors...")
src_list, tgt_list = [], []
for k, e in zip(kor_corpus, eng_corpus):
    src_ids = ko_tokenizer.encode_as_ids(k)
    tgt_ids = en_tokenizer.encode_as_ids(e)
    if 1 <= len(src_ids) <= 50 and 1 <= len(tgt_ids) <= 50:
        src_list.append(torch.tensor(src_ids))
        tgt_list.append(torch.tensor(tgt_ids))

enc_train = torch.nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=0)
dec_train = torch.nn.utils.rnn.pad_sequence(tgt_list, batch_first=True, padding_value=0)

print(f'Preprocessing complete. enc_train shape: {enc_train.shape}')

[ ]
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# 1. Split data into training and validation sets (8:2)
BATCH_SIZE = 64
enc_train_split, enc_val_split, dec_train_split, dec_val_split = train_test_split(
    enc_train, dec_train, test_size=0.2, random_state=42
)

# 2. Create Dataset and DataLoader
# Target input: everything except the last token (<EOS>)
# Target output: everything except the first token (<BOS>)
train_dataset = TensorDataset(
    enc_train_split,
    dec_train_split[:, :-1],
    dec_train_split[:, 1:]
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(
    enc_val_split,
    dec_val_split[:, :-1],
    dec_val_split[:, 1:]
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders initialized!")
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

[ ]
# 학습용 헬퍼 함수: 배치를 입력받아 마스크를 포함해 모델을 실행
def train_step(model, batch, optimizer, criterion, device):
    model.train()
    src, tgt_in, tgt_out = [b.to(device) for b in batch]

    # 마스크 생성 및 장치 이동
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    optimizer.zero_grad()

    # Forward Pass
    logits, _, _, _ = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # Loss 계산 (Padding 토큰은 무시하도록 설정됨)
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))

    loss.backward()
    optimizer.step()

    return loss.item()

[ ]
import torch.optim as optim
import torch.nn as nn

# 1. Warmup Scheduler 정의
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = max(1.0, float(step))
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

# 2. 손실 함수 (Loss Function) 설정
# ignore_index=0: 패딩 토큰에 의한 손실은 계산에서 제외합니다.
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 3. 스케줄러 및 옵티마이저 설정
D_MODEL = 512
warmup_steps = 4000
lr_scheduler = LearningRateScheduler(D_MODEL, warmup_steps)

# 초기 lr은 warmup 스케줄러를 통해 계산된 값으로 시작
optimizer = optim.Adam(
    transformer.parameters(),
    lr=lr_scheduler(1),
    betas=(0.9, 0.98),
    eps=1e-9
)

print("손실 함수 및 옵티마이저 설정 완료!")
🚀 번역(Inference) 테스트를 위한 함수 구현
모델의 학습이 완료된 후, 새로운 한국어 문장을 입력받아 영어로 번역해주는 predict 함수를 정의합니다.


[ ]
def predict(sentence, model, src_tokenizer, tgt_tokenizer, device, max_len=50):
    model.eval()

    # 1. 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)
    src_input = torch.tensor([tokens], dtype=torch.long).to(device)

    # 2. 인코더 마스크 생성
    enc_mask = generate_padding_mask(src_input)

    # 3. 디코더 입력 초기화 (<BOS> 토큰으로 시작)
    # tgt_tokenizer.bos_id() 가 1로 설정되어 있습니다.
    output = torch.tensor([[1]], dtype=torch.long).to(device)

    with torch.no_grad():
        for i in range(max_len):
            # 현재까지 생성된 시퀀스에 대한 마스크 생성
            _, dec_enc_mask, dec_mask = generate_masks(src_input, output)

            # Forward Pass
            logits, _, _, _ = model(
                src_input, output, enc_mask, dec_enc_mask, dec_mask
            )

            # 마지막 타임스텝의 결과만 선택하여 다음 단어 예측
            # [batch, seq, vocab] -> [1, -1, vocab]
            prediction = logits[:, -1, :].argmax(dim=-1).item()

            # 예측된 단어를 결과 시퀀스에 추가
            next_token = torch.tensor([[prediction]], dtype=torch.long).to(device)
            output = torch.cat([output, next_token], dim=1)

            # <EOS> 토큰(2)이 나오면 종료
            if prediction == 2:
                break

    # 4. 토큰을 문장으로 디코딩
    ids = output.squeeze().tolist()
    result = tgt_tokenizer.decode_ids(ids)

    return result

print("추론(predict) 함수 정의 완료!")
💾 모델 가중치 저장 및 불러오기
학습이 완료된 모델의 파라미터를 저장하고, 필요할 때 다시 로드하여 추론에 사용할 수 있습니다.


[ ]
# 현재 메모리의 모델 상태를 저장하여 이름 불일치 해결
save_path = "transformer_final.pth"
torch.save(transformer.state_dict(), save_path)

# 다시 로드 테스트
transformer.load_state_dict(torch.load(save_path, map_location=device))
transformer.eval()

print("가중치를 현재 구조에 맞춰 동기화하고 성공적으로 불러왔습니다!")

[ ]
# 번역 테스트 예시
test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."

# 주의: 실제 학습이 어느 정도 진행된 후에 유의미한 결과가 나옵니다.
# translation = predict(test_sentence, transformer, ko_tokenizer, en_tokenizer, device)
# print(f"입력: {test_sentence}")
# print(f"번역: {translation}")
훈련하기


[ ]
def positional_encoding(pos, d_model):
    def cal_angle(position, i):

        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

[ ]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = K.shape[-1]
        QK = torch.matmul(Q, K.transpose(-2, -1))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask == 0, float('-1e9'))

        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)

        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        out = self.combine_heads(out)
        out = self.linear(out)

        return out, attention_weights

[ ]
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)

        return out

[ ]
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn

[ ]
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        """
        Masked Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, padding_mask)
        out = self.dropout(out)
        out += residual

        """
        Multi-Head Attention
        """
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, causality_mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn

[ ]
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = x
        enc_attns = []

        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        return out, enc_attns

[ ]
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

    def forward(self, x, enc_out, causality_mask, padding_mask):
        out = x

        dec_attns = []
        dec_enc_attns = []

        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, causality_mask, padding_mask
            )

            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        return out, dec_attns, dec_enc_attns

[ ]
class Transformer(nn.Module):
    def __init__(self,
                 n_layers,
                 d_model,
                 n_heads,
                 d_ff,
                 src_vocab_size,
                 tgt_vocab_size,
                 pos_len,
                 dropout=0.2,
                 shared=True):

        super(Transformer, self).__init__()

        self.d_model = d_model
        self.shared = shared

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # numpy array를 torch tensor로 변환하여 저장
        pos_enc = positional_encoding(pos_len, d_model)
        self.pos_encoding = torch.tensor(pos_enc, dtype=torch.float32)

        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        if shared:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.shape[1]

        out = emb(x)

        if self.shared:
            out *= torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))

        # 장치(device)에 맞춰서 위치 인코딩 더하기
        pos_enc = self.pos_encoding[:seq_len, :].unsqueeze(0).to(x.device)
        out += pos_enc

        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)

        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, causality_mask, dec_mask)

        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns

[ ]
def generate_padding_mask(seq):
    """ 패딩된 부분(0)을 1로 변환하여 마스크 생성 """
    mask = (seq == 0).float()
    return mask[:, None, None, :]

def generate_causality_mask(src_len, tgt_len):
    """ 미래 정보를 참조하지 않도록 Causal Mask 생성 """
    mask = 1 - torch.cumsum(torch.eye(src_len, tgt_len), dim=0)
    return mask.float()

def generate_masks(src, tgt):
    """ Encoder-Decoder에서 사용할 마스크 생성 """
    enc_mask = generate_padding_mask(src)
    dec_mask = generate_padding_mask(tgt)

    dec_causality_mask = generate_causality_mask(tgt.shape[1], tgt.shape[1])
    dec_mask = torch.max(dec_mask, dec_causality_mask.to(dec_mask.device))

    dec_enc_causality_mask = generate_causality_mask(tgt.shape[1], src.shape[1])
    dec_enc_mask = torch.max(enc_mask, dec_enc_causality_mask.to(enc_mask.device))

    return enc_mask, dec_enc_mask, dec_mask

[ ]
# 단어 사전 크기 정의
SRC_VOCAB_SIZE = 20000
TGT_VOCAB_SIZE = 20000

# Transformer 모델 초기화
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=200,
    dropout=0.2,
    shared=True
)

[ ]
import math

class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

[ ]
learning_rate = LearningRateScheduler(optimizer, d_model=512)

optimizer = torch.optim.Adam(optimizer.param_groups,
                             lr=learning_rate.get_lr()[0],
                             betas=(0.9, 0.98),
                             eps=1e-9)

[ ]
loss_object = torch.nn.CrossEntropyLoss(reduction='none')

def loss_function(real, pred):
    mask = (real != 0)
    loss_ = loss_object(pred, real)

    # Masking 되지 않은 입력의 개수로 Scaling하는 과정
    mask = mask.float()
    loss_ *= mask

    return loss_.sum() / mask.sum()

[ ]
# Train Step 함수 정의
def train_step(src, tgt, model, optimizer):
    gold = tgt[:, 1:]

    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt)

    # 계산된 loss에 대해 역전파(Backpropagation)를 적용해 학습을 진행합니다.
    optimizer.zero_grad()
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt, enc_mask, dec_enc_mask, dec_mask)
    loss = loss_function(gold, predictions[:, :-1])

    loss.backward()

    # 최종적으로 optimizer.step()이 사용됩니다.
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns


[ ]
# Attention 시각화 함수

def visualize_attention(src, tgt, enc_attns, dec_attns, dec_enc_attns):
    def draw(data, ax, x="auto", y="auto"):
        import seaborn
        seaborn.heatmap(data,
                        square=True,
                        vmin=0.0, vmax=1.0,
                        cbar=False, ax=ax,
                        xticklabels=x,
                        yticklabels=y)

    for layer in range(0, 2, 1):
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Encoder Layer", layer + 1)
        for h in range(4):
            draw(enc_attns[layer][0, h, :len(src), :len(src)], axs[h], src, src)
        plt.show()

    for layer in range(0, 2, 1):
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Decoder Self Layer", layer+1)
        for h in range(4):
            draw(dec_attns[layer][0, h, :len(tgt), :len(tgt)], axs[h], tgt, tgt)
        plt.show()

        print("Decoder Src Layer", layer+1)
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        for h in range(4):
            draw(dec_enc_attns[layer][0, h, :len(tgt), :len(src)], axs[h], src, tgt)
        plt.show()

[ ]
# 번역 생성 함수 (디버깅 코드 추가)
def evaluate(sentence, model, src_tokenizer, tgt_tokenizer):
    model.eval()
    device = next(model.parameters()).device

    sentence = preprocess_sentence(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)
    _input = torch.tensor(tokens).unsqueeze(0).to(device)

    ids = []
    output = torch.tensor([tgt_tokenizer.bos_id()]).unsqueeze(0).to(device)

    print(f"[Debug] Start inference. BOS ID: {tgt_tokenizer.bos_id()}")

    for i in range(dec_train.shape[-1]):
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(_input, output)

        predictions, enc_attns, dec_attns, dec_enc_attns = \
        model(_input, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 타임스텝의 로짓에서 가장 확률 높은 단어 선택
        predicted_id = torch.argmax(predictions[0, -1]).item()

        if i < 5: # 초기 5개 토큰 로그 출력
            print(f"Step {i}: Predicted ID {predicted_id}")

        if tgt_tokenizer.eos_id() == predicted_id:
            print(f"[Debug] EOS detected at step {i}")
            break

        ids.append(predicted_id)
        output = torch.cat([output, torch.tensor([[predicted_id]]).to(device)], dim=-1)

    result = tgt_tokenizer.decode_ids(ids)
    return None, result, enc_attns, dec_attns, dec_enc_attns

[ ]
# 번역 생성 및 Attention 시각화 결합

def translate(sentence, model, src_tokenizer, tgt_tokenizer, plot_attention=False):
    pieces, result, enc_attns, dec_attns, dec_enc_attns = \
    evaluate(sentence, model, src_tokenizer, tgt_tokenizer)

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    if plot_attention:
        visualize_attention(pieces, result.split(), enc_attns, dec_attns, dec_enc_attns)

[ ]
# 번역 테스트 다시 수행
test_sentence = "They are in the city."

print("--- 번역 결과 확인 ---")
translate(test_sentence, transformer, ko_tokenizer, en_tokenizer, plot_attention=False)

[ ]
# 학습 건전성 테스트 (1 Step 실행)
transformer.train()
src, tgt_in, tgt_out = next(iter(train_loader))
src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
logits, _, _, _ = transformer(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))

print(f"현재 배치 손실값: {loss.item():.4f}")
if loss.item() > 10:
    print("경고: 손실값이 매우 높습니다. 모델이 초기화 상태이거나 학습이 필요합니다.")

[ ]
# 특정 레이어의 가중치 상태 확인
with torch.no_grad():
    weight_sample = transformer.encoder.enc_layers[0].enc_self_attn.W_q.weight
    print(f"Layer: encoder.enc_layers[0].enc_self_attn.W_q.weight")
    print(f"Mean: {weight_sample.mean().item():.8f}")
    print(f"Std:  {weight_sample.std().item():.8f}")
    print(f"Max:  {weight_sample.max().item():.8f}")

# 가중치가 모두 0에 가깝거나 초기 분포(예: Xavier/Kaiming)와 일치하는지 확인하기 위함입니다.

[ ]
# 1. 학습 환경 재설정 (학습률 하향 및 마스크 확인)
optimizer = optim.Adam(transformer.parameters(), lr=0.00005, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 2. 아주 적은 데이터로 학습이 되는지 먼저 테스트 (Overfitting Test)
transformer.train()
src, tgt_in, tgt_out = next(iter(train_loader))
src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

print("테스트 학습 시작...")
for i in range(100):
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    optimizer.zero_grad()
    logits, _, _, _ = transformer(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
    loss.backward()
    optimizer.step()

    if (i+1) % 20 == 0:
        print(f"Step {i+1}, Loss: {loss.item():.4f}")

print("\n학습 후 동일 문장에 대해 번역 시도:")
transformer.eval()
with torch.no_grad():
    # 첫 번째 샘플 문장(src[0])에 대해 예측
    test_src = src[0:1]
    test_tgt_in = torch.tensor([[1]], device=device) # <BOS>만 입력

    # 간단한 greedy search 루프
    for _ in range(10):
        enc_m, dec_enc_m, dec_m = generate_masks(test_src, test_tgt_in)
        pred, _, _, _ = transformer(test_src, test_tgt_in, enc_m.to(device), dec_enc_m.to(device), dec_m.to(device))
        next_word = pred[:, -1, :].argmax(dim=-1).unsqueeze(1)
        test_tgt_in = torch.cat([test_tgt_in, next_word], dim=1)
        if next_word.item() == 2: break # <EOS>

print(f"결과 토큰 ID: {test_tgt_in.squeeze().tolist()}")
print(f"디코딩 결과: {en_tokenizer.decode_ids(test_tgt_in.squeeze().tolist())}")

[ ]
# 과적합 테스트 연장 (500 Step)
transformer.train()
src, tgt_in, tgt_out = next(iter(train_loader))
src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

print("--- 500 Step Overfitting Test Start ---")
for i in range(500):
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    optimizer.zero_grad()
    logits, _, _, _ = transformer(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
    loss.backward()
    optimizer.step()

    if (i+1) % 100 == 0:
        print(f"Step {i+1}, Loss: {loss.item():.4f}")

print("\n--- Inference Test (Greedy Search) ---")
transformer.eval()
with torch.no_grad():
    test_src = src[0:1]
    test_tgt_in = torch.tensor([[1]], device=device)

    for _ in range(20):
        enc_m, dec_enc_m, dec_m = generate_masks(test_src, test_tgt_in)
        pred, _, _, _ = transformer(test_src, test_tgt_in, enc_m.to(device), dec_enc_m.to(device), dec_m.to(device))
        next_word = pred[:, -1, :].argmax(dim=-1).unsqueeze(1)
        test_tgt_in = torch.cat([test_tgt_in, next_word], dim=1)
        if next_word.item() == 2: break

print(f"Predicted Token IDs: {test_tgt_in.squeeze().tolist()}")
print(f"Decoded Translation: {en_tokenizer.decode_ids(test_tgt_in.squeeze().tolist())}")

[ ]
# 번역 과정 디버깅: 모델이 예측하는 토큰 ID를 직접 확인
def debug_predict(sentence, model, src_tokenizer, tgt_tokenizer, device, max_len=20):
    model.eval()
    sentence = preprocess_sentence(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)
    src_input = torch.tensor([tokens], dtype=torch.long).to(device)

    # <BOS> 토큰으로 시작 (ID: 1)
    output = torch.tensor([[1]], dtype=torch.long).to(device)

    print(f"[입력 문장]: {sentence}")
    print(f"[입력 토큰]: {tokens}")

    results = []
    with torch.no_grad():
        for i in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src_input, output)
            logits, _, _, _ = model(src_input, output, enc_mask, dec_enc_mask, dec_mask)

            # 마지막 타임스텝의 예측값
            probs = logits[:, -1, :]
            prediction = probs.argmax(dim=-1).item()

            print(f"Step {i+1}: Predicted Token ID = {prediction}")

            if prediction == 2: # <EOS>
                print("Reached <EOS>")
                break

            results.append(prediction)
            next_token = torch.tensor([[prediction]], dtype=torch.long).to(device)
            output = torch.cat([output, next_token], dim=1)

    decoded = tgt_tokenizer.decode_ids(results)
    return decoded

# 디버깅 실행
print("--- 추론 디버깅 시작 ---")
debug_result = debug_predict("시민들은 그 결정에 반대하며 거리에 모였습니다.", transformer, ko_tokenizer, en_tokenizer, device)
print(f"\n[최종 디코딩 결과]: {debug_result}")

[ ]
# 최종 번역 테스트 및 결과 확인
test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."

print("--- 번역 테스트 시작 ---")
_, result, _, _, _ = evaluate(test_sentence, transformer, ko_tokenizer, en_tokenizer)
print(f"\n입력 문장: {test_sentence}")
print(f"번역 결과: {result}")

[ ]
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

# 0. 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 트랜스포머 하위 모듈 정의
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)
    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)
    def split_heads(self, x):
        batch_size = x.size(0)
        return x.view(batch_size, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        q, k, v = self.W_q(Q), self.W_k(K), self.W_v(V)
        q, k, v = self.split_heads(q), self.split_heads(k), self.split_heads(v)
        d_k = k.size(-1)
        scaled_qk = torch.matmul(q, k.transpose(-2, -1)) / (d_k**0.5)
        if mask is not None: scaled_qk += (mask * -1e9)
        attn = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attn, v).permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)
        return self.linear(out), attn

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.w_2(F.relu(self.w_1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask):
        attn_output, attn = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, n_heads)
        self.mha2 = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, causality_mask, padding_mask):
        out_mha1, self_attn = self.mha1(x, x, x, causality_mask)
        x = self.norm1(x + self.dropout(out_mha1))
        out_mha2, dec_enc_attn = self.mha2(x, enc_out, enc_out, padding_mask)
        x = self.norm2(x + self.dropout(out_mha2))
        out_ffn = self.ffn(x)
        x = self.norm3(x + self.dropout(out_ffn))
        return x, self_attn, dec_enc_attn

class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
    def forward(self, x, mask):
        attns = []
        for layer in self.layers: x, attn = layer(x, mask); attns.append(attn)
        return x, attns

class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
    def forward(self, x, enc_out, causality_mask, padding_mask):
        self_attns, dec_enc_attns = [], []
        for layer in self.layers: x, sa, dea = layer(x, enc_out, causality_mask, padding_mask); self_attns.append(sa); dec_enc_attns.append(dea)
        return x, self_attns, dec_enc_attns

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, src_vocab_size, tgt_vocab_size, pos_len, dropout=0.2, shared=True):
        super().__init__()
        self.d_model = float(d_model)
        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = torch.tensor(positional_encoding(pos_len, d_model), dtype=torch.float32)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        if shared: self.fc.weight = self.dec_emb.weight
    def embedding(self, emb, x):
        seq_len = x.size(1)
        out = emb(x) * (self.d_model**0.5)
        out += self.pos_encoding[:seq_len, :].to(x.device).unsqueeze(0)
        return self.dropout(out)
    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_in, dec_in = self.embedding(self.enc_emb, enc_in), self.embedding(self.dec_emb, dec_in)
        enc_out, enc_attns = self.encoder(enc_in, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, dec_mask, dec_enc_mask)
        return self.fc(dec_out), enc_attns, dec_attns, dec_enc_attns

[ ]
# Validation Test with Dummy Tensors
# This script verifies that the Transformer returns 4 values and that attention maps have the correct dimensions

# 1. Hyperparameters for testing
BATCH_SIZE = 4
SEQ_LEN_SRC = 10
SEQ_LEN_TGT = 12
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 2
SRC_VOCAB = 100
TGT_VOCAB = 100
POS_LEN = 20

# 2. Instantiate Model
test_model = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB,
    tgt_vocab_size=TGT_VOCAB,
    pos_len=POS_LEN
).to(device)

# 3. Create Dummy Inputs and Masks
src_dummy = torch.randint(0, SRC_VOCAB, (BATCH_SIZE, SEQ_LEN_SRC)).to(device)
tgt_dummy = torch.randint(0, TGT_VOCAB, (BATCH_SIZE, SEQ_LEN_TGT)).to(device)

# Masks: (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
e_mask = torch.zeros(BATCH_SIZE, 1, 1, SEQ_LEN_SRC).to(device)
de_mask = torch.zeros(BATCH_SIZE, 1, 1, SEQ_LEN_SRC).to(device)
d_mask = torch.zeros(BATCH_SIZE, 1, SEQ_LEN_TGT, SEQ_LEN_TGT).to(device)

# 4. Forward Pass
logits, enc_attns, dec_attns, dec_enc_attns = test_model(src_dummy, tgt_dummy, e_mask, de_mask, d_mask)

# 5. Verify Shapes
print(f"Logits shape: {logits.shape} (Expected: [{BATCH_SIZE}, {SEQ_LEN_TGT}, {TGT_VOCAB}])")
print(f"Number of Encoder layers: {len(enc_attns)}")
print(f"Encoder Attention shape: {enc_attns[0].shape} (Expected: [{BATCH_SIZE}, {N_HEADS}, {SEQ_LEN_SRC}, {SEQ_LEN_SRC}])")
print(f"Decoder Self-Attention shape: {dec_attns[0].shape} (Expected: [{BATCH_SIZE}, {N_HEADS}, {SEQ_LEN_TGT}, {SEQ_LEN_TGT}])")
print(f"Cross-Attention shape: {dec_enc_attns[0].shape} (Expected: [{BATCH_SIZE}, {N_HEADS}, {SEQ_LEN_TGT}, {SEQ_LEN_SRC}])")

assert logits.shape == (BATCH_SIZE, SEQ_LEN_TGT, TGT_VOCAB)
assert len(enc_attns) == N_LAYERS
assert enc_attns[0].shape == (BATCH_SIZE, N_HEADS, SEQ_LEN_SRC, SEQ_LEN_SRC)
print("\nShape verification successful!")
Logits shape: torch.Size([4, 12, 100]) (Expected: [4, 12, 100])
Number of Encoder layers: 2
Encoder Attention shape: torch.Size([4, 8, 10, 10]) (Expected: [4, 8, 10, 10])
Decoder Self-Attention shape: torch.Size([4, 8, 12, 12]) (Expected: [4, 8, 12, 12])
Cross-Attention shape: torch.Size([4, 8, 12, 10]) (Expected: [4, 8, 12, 10])

Shape verification successful!

[ ]
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 학습 손실 시각화
def plot_loss(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss', color='blue')
    plt.plot(val_losses, label='Validation Loss', color='red')
    plt.title('Transformer Training Progress')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_loss(train_losses, val_losses)

[ ]
# 2. Attention Map 시각화 함수 정의 및 마스킹/토크나이저 복구
import sentencepiece as spm
import os
import re
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z\u200B\u0020\uAC00-\uD7AF?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[\" \"]+', " ", sentence)
    return sentence.strip()

def generate_padding_mask(seq):
    mask = (seq == 0).float()
    return mask.unsqueeze(1).unsqueeze(2)

def generate_causality_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    tgt_seq_len = tgt.shape[1]
    causal_mask = generate_causality_mask(tgt_seq_len).to(tgt.device)
    dec_padding_mask = generate_padding_mask(tgt).to(tgt.device)
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)
    return enc_mask, dec_enc_mask, dec_mask

def visualize_attention(model, sentence, src_tokenizer, tgt_tokenizer, device):
    plt.rc('font', family='DejaVu Sans')
    model.eval()

    sentence = preprocess_sentence(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)
    src_input = torch.tensor([tokens], dtype=torch.long).to(device)

    output = torch.tensor([[1]], dtype=torch.long).to(device) # <BOS>
    final_dec_enc_attns = None
    vocab_size = tgt_tokenizer.get_piece_size()

    with torch.no_grad():
        for i in range(20):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src_input, output)
            logits, enc_attns, dec_attns, dec_enc_attns = model(src_input, output, enc_mask, dec_enc_mask, dec_mask)

            prediction = logits[:, -1, :].argmax(dim=-1).item()
            if prediction == 2: break # <EOS>

            next_token = torch.tensor([[prediction]], dtype=torch.long).to(device)
            output = torch.cat([output, next_token], dim=1)
            final_dec_enc_attns = dec_enc_attns

    result_tokens = output.squeeze().tolist()
    if isinstance(result_tokens, int): result_tokens = [result_tokens]
    # Exclude BOS (1) from labels
    valid_tokens = [t for t in result_tokens if 0 <= t < vocab_size and t != 1]
    result_sentence = tgt_tokenizer.decode_ids(valid_tokens)

    src_labels = [src_tokenizer.id_to_piece(t) for t in tokens] + ["<EOS>"]
    tgt_labels = [tgt_tokenizer.id_to_piece(t) for t in valid_tokens]

    print(f"입력: {sentence}")
    print(f"번역: {result_sentence if result_sentence else '[결과 없음 - 학습 필요]'}")

    if final_dec_enc_attns is not None:
        # attention shape: [Heads, Tgt_Seq, Src_Seq]
        attention = final_dec_enc_attns[-1][0, 0].cpu().numpy()

        h = len(tgt_labels)
        w = len(src_labels)

        if h == 0:
            # Prevent ValueError by creating a dummy row if no output tokens generated
            plot_data = np.zeros((1, w))
            y_ticks = ['(No Output)']
        else:
            plot_data = attention[:h, :w]
            y_ticks = tgt_labels

        plt.figure(figsize=(10, 8))
        sns.heatmap(plot_data,
                    xticklabels=src_labels,
                    yticklabels=y_ticks,
                    annot=False, cmap='viridis')
        plt.title("Attention Map (Decoder-Encoder Cross-Attention)")
        plt.xlabel("Source (Korean)")
        plt.ylabel("Target (Generated English)")
        plt.show()
    else:
        print("\n[오류] 모델로부터 어텐션 가중치를 가져오지 못했습니다.")

test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."
visualize_attention(transformer, test_sentence, ko_tokenizer, en_tokenizer, device)

[ ]
# 50 에포크 대폭 학습 시작
import torch
from tqdm.notebook import tqdm

# train_loader와 val_loader가 정의되어 있는지 확인
if 'train_loader' not in globals():
    raise NameError("train_loader가 정의되지 않았습니다. 데이터 전처리 및 DataLoader 생성 셀(예: fbcec17e)을 먼저 실행해 주세요.")

EPOCHS = 50
learning_rate = 0.0001
optimizer = torch.optim.Adam(transformer.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

train_losses = []
val_losses = []

print(f"{device} 장치에서 {EPOCHS} 에포크 학습을 시작합니다.")

for epoch in range(EPOCHS):
    transformer.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        # train_step 함수가 정의되어 있어야 합니다.
        loss = train_step(transformer, batch, optimizer, criterion, device)
        total_train_loss += loss

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = evaluate_validation(transformer, val_loader, criterion, device)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} 완료 | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # 5 에포크마다 성능 확인
    if (epoch + 1) % 5 == 0:
        test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."
        translation = predict(test_sentence, transformer, ko_tokenizer, en_tokenizer, device)
        print(f"[중간 결과] 번역: {translation}")

# 모델 저장
torch.save(transformer.state_dict(), "transformer_boosted_model.pth")
print("전체 학습이 완료되고 모델이 저장되었습니다!")

[ ]
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import re
import torch
import sentencepiece as spm

# 1. Utility function for preprocessing
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[\" \"]+', " ", sentence)
    return sentence

# 2. 데이터 디렉토리 설정 및 데이터 다운로드
data_dir = os.path.join(os.getenv("HOME"), 'work/Transformer/data')
kor_path = os.path.join(data_dir, "korean-english-park.train.ko")
eng_path = os.path.join(data_dir, "korean-english-park.train.en")

if not os.path.exists(kor_path):
    print("Data files not found. Downloading and extracting...")
    os.makedirs(data_dir, exist_ok=True)
    !wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz -P {data_dir}
    !tar -xvf {data_dir}/korean-english-park.train.tar.gz -C {data_dir}

# 3. 데이터 로드 및 정제
def clean_corpus(kor_path, eng_path):
    with open(kor_path, "r", encoding='utf-8') as f: kor = f.read().splitlines()
    with open(eng_path, "r", encoding='utf-8') as f: eng = f.read().splitlines()
    cleaned = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))
    return cleaned

cleaned_corpus = clean_corpus(kor_path, eng_path)
kor_corpus = []
eng_corpus = []
for pair in cleaned_corpus:
    parts = pair.split("\t")
    if len(parts) == 2:
        k, e = parts
        kor_corpus.append(preprocess_sentence(k))
        eng_corpus.append(preprocess_sentence(e))

# 4. 토크나이저 생성
def generate_tokenizer(corpus, lang="ko", vocab_size=20000):
    file = f"./{lang}_corpus.txt"
    model_prefix = f"{lang}_spm"
    with open(file, 'w', encoding='utf-8') as f:
        for row in corpus: f.write(str(row) + '\n')
    spm.SentencePieceTrainer.Train(f'--input={file} --model_prefix={model_prefix} --vocab_size={vocab_size} --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3')
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load(f'{model_prefix}.model')
    return tokenizer

ko_tokenizer = generate_tokenizer(kor_corpus, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, "en")
en_tokenizer.set_encode_extra_options("bos:eos")

# 5. 토큰화 및 패딩
src_corpus_ids = []
tgt_corpus_ids = []
for idx in range(len(kor_corpus)):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])
    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus_ids.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus_ids.append(torch.tensor(tgt_tokens, dtype=torch.long))

enc_train = torch.nn.utils.rnn.pad_sequence(src_corpus_ids, batch_first=True, padding_value=0)
dec_train = torch.nn.utils.rnn.pad_sequence(tgt_corpus_ids, batch_first=True, padding_value=0)

# 6. DataLoader 설정
BATCH_SIZE = 64
enc_train_split, enc_val_split, dec_train_split, dec_val_split = train_test_split(enc_train, dec_train, test_size=0.2, random_state=42)

train_dataset = TensorDataset(enc_train_split, dec_train_split[:, :-1], dec_train_split[:, 1:])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(enc_val_split, dec_val_split[:, :-1], dec_val_split[:, 1:])
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders initialized: {len(train_loader)} train batches.")
Data files not found. Downloading and extracting...
--2026-05-13 05:17:58--  https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz [following]
--2026-05-13 05:17:59--  https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8718893 (8.3M) [application/octet-stream]
Saving to: ‘/root/work/Transformer/data/korean-english-park.train.tar.gz’

korean-english-park 100%[===================>]   8.31M  --.-KB/s    in 0.06s

2026-05-13 05:18:00 (128 MB/s) - ‘/root/work/Transformer/data/korean-english-park.train.tar.gz’ saved [8718893/8718893]

korean-english-park.train.en
korean-english-park.train.ko
DataLoaders initialized: 902 train batches.

[ ]
# 학습 실행 전 필수 변수 존재 여부 확인
required_vars = [
    'train_loader', 'val_loader', 'ko_tokenizer', 'en_tokenizer',
    'transformer', 'optimizer', 'criterion', 'device', 'train_step', 'evaluate_validation'
]

missing_vars = []
for var in required_vars:
    if var not in globals():
        missing_vars.append(var)

if not missing_vars:
    print("✅ 모든 학습 준비가 완료되었습니다! Cell fa9e2b05를 실행하여 학습을 시작하세요.")
    print(f"- 학습 데이터 배치 수: {len(train_loader)}")
    print(f"- 검증 데이터 배치 수: {len(val_loader)}")
    print(f"- 사용 장치: {device}")
else:
    print(f"❌ 다음 변수들이 누락되었습니다: {missing_vars}")
    print("데이터 로드 및 모델 정의 셀(fbcec17e, fa9e2b05 등)을 다시 확인해 주세요.")

[ ]
def generate_padding_mask(seq):
    mask = (seq == 0).float()
    return mask.unsqueeze(1).unsqueeze(2)

def generate_causality_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    tgt_seq_len = tgt.shape[1]
    causal_mask = generate_causality_mask(tgt_seq_len).to(tgt.device)
    dec_padding_mask = generate_padding_mask(tgt).to(tgt.device)
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)
    return enc_mask, dec_enc_mask, dec_mask

def train_step(model, batch, optimizer, criterion, device):
    model.train()
    src, tgt_in, tgt_out = [b.to(device) for b in batch]
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    optimizer.zero_grad()
    logits, _, _, _ = model(src, tgt_in, enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device))
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate_validation(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            src, tgt_in, tgt_out = [b.to(device) for b in batch]
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
            logits, _, _, _ = model(src, tgt_in, enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device))
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

print("Masking and helper functions redefined.")

[ ]
import torch
from tqdm.notebook import tqdm

# Helper functions to ensure they are defined in scope
def generate_padding_mask(seq):
    mask = (seq == 0).float()
    return mask.unsqueeze(1).unsqueeze(2)

def generate_causality_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    tgt_seq_len = tgt.shape[1]
    causal_mask = generate_causality_mask(tgt_seq_len).to(tgt.device)
    dec_padding_mask = generate_padding_mask(tgt).to(tgt.device)
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)
    return enc_mask, dec_enc_mask, dec_mask

def train_step(model, batch, optimizer, criterion, device):
    model.train()
    src, tgt_in, tgt_out = [b.to(device) for b in batch]
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
    optimizer.zero_grad()
    logits, _, _, _ = model(src, tgt_in, enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device))
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate_validation(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            src, tgt_in, tgt_out = [b.to(device) for b in batch]
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
            logits, _, _, _ = model(src, tgt_in, enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device))
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 50
print(f"{device} device is starting {EPOCHS} epochs of training.")

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        loss = train_step(transformer, batch, optimizer, criterion, device)
        total_train_loss += loss

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = evaluate_validation(transformer, val_loader, criterion, device)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} Complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        try:
            test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."
            translation = predict(test_sentence, transformer, ko_tokenizer, en_tokenizer, device)
            print(f"[Intermediate Result] Input: {test_sentence}")
            print(f"[Intermediate Result] Translation: {translation}")
        except NameError:
            print("[Info] predict function not found, skipping intermediate test.")

torch.save(transformer.state_dict(), "transformer_boosted_model.pth")
print("Training complete! Model saved as 'transformer_boosted_model.pth'.")

[ ]
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

# --- 1. Helper Functions for Masking & Training ---
def generate_padding_mask(seq):
    mask = (seq == 0).float()
    return mask.unsqueeze(1).unsqueeze(2)

def generate_causality_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)
    tgt_seq_len = tgt.shape[1]
    causal_mask = generate_causality_mask(tgt_seq_len).to(tgt.device)
    dec_padding_mask = generate_padding_mask(tgt).to(tgt.device)
    dec_mask = torch.maximum(causal_mask.unsqueeze(0).unsqueeze(1), dec_padding_mask)
    return enc_mask, dec_enc_mask, dec_mask

def train_step(model, batch, optimizer, criterion, device):
    model.train()
    src, tgt_in, tgt_out = [b.to(device) for b in batch]
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    optimizer.zero_grad()
    logits, _, _, _ = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
    loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate_validation(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            src, tgt_in, tgt_out = [b.to(device) for b in batch]
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
            logits, _, _, _ = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
            loss = criterion(logits.view(-1, logits.size(-1)), tgt_out.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

# --- 2. Training Configuration ---
EPOCHS = 50
learning_rate = 0.0001
# Use the transformer instance defined earlier
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transformer.to(device)

optimizer = optim.Adam(transformer.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)

train_losses = []
val_losses = []

print(f"Starting training on {device} for {EPOCHS} epochs.")

# --- 3. Training Loop ---
for epoch in range(EPOCHS):
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        loss = train_step(transformer, batch, optimizer, criterion, device)
        total_train_loss += loss

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = evaluate_validation(transformer, val_loader, criterion, device)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f'Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    # Test translation every 5 epochs
    if (epoch + 1) % 5 == 0:
        test_sentence = "시민들은 그 결정에 반대하며 거리에 모였습니다."
        try:
            res = predict(test_sentence, transformer, ko_tokenizer, en_tokenizer, device)
            print(f'\n[Sample Prediction] {test_sentence} -> {res}\n')
        except NameError:
            pass

torch.save(transformer.state_dict(), 'transformer_boosted_model.pth')
print("Training complete! Model saved.")

🧪 학습된 모델 로드 및 번역 테스트
저장된 체크포인트를 불러와서 실제 번역 성능을 확인합니다.


[ ]
import torch

# 1. 모델 가중치 로드
model_path = 'transformer_boosted_model.pth'
transformer.load_state_dict(torch.load(model_path, map_location=device))
transformer.eval()
print(f"모델 '{model_path}' 로드 완료!")

# 2. 번역 테스트 수행
test_sentences = [
    "시민들은 그 결정에 반대하며 거리에 모였습니다.",
    "그는 내일 학교에 갈 것입니다.",
    "오늘 날씨가 매우 화창합니다."
]

print("\n--- [번역 테스트 결과] ---")
for sen in test_sentences:
    try:
        translation = predict(sen, transformer, ko_tokenizer, en_tokenizer, device)
        print(f"입력: {sen}")
        print(f"출력: {translation}\n")
    except Exception as e:
        print(f"오류 발생 ({sen}): {e}")
🏗️ 트랜스포머 모델 구조 정의
이 섹션에서는 트랜스포머를 구성하는 핵심 컴포넌트들을 PyTorch 클래스로 정의합니다.


[ ]
from google.colab import drive
drive.mount('/content/drive')
Your Google Drive is now mounted. Please replace path/to/your/data_file.csv with the actual path to your data file in Google Drive. For example, if your file is named my_data.csv and is located in the root of 'My Drive', the path would be /content/drive/My Drive/my_data.csv.


[ ]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, q, k, v, mask):
        batch_size = q.size(0)

        q = self.split_heads(self.W_q(q), batch_size)
        k = self.split_heads(self.W_k(k), batch_size)
        v = self.split_heads(self.W_v(v), batch_size)

        # Scaled Dot-Product Attention
        scaled_attention_logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.depth)
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = F.softmax(scaled_attention_logits, dim=-1)
        output = torch.matmul(attention_weights, v)

        output = output.permute(0, 2, 1, 3).contiguous()
        concat_attention = output.view(batch_size, -1, self.d_model)

        return self.linear(concat_attention), attention_weights

[ ]
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)

    def forward(self, x, mask):
        attn_output, _ = self.mha(x, x, x, mask)
        x = self.layernorm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(x)
        x = self.layernorm2(x + self.dropout2(ffn_output))
        return x

[ ]
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)
        self.dropout3 = nn.Dropout(rate)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        # 1. Masked Self-Attention
        attn1, _ = self.mha1(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(attn1 + x)

        # 2. Encoder-Decoder Attention
        attn2, _ = self.mha2(out1, enc_output, enc_output, padding_mask)
        out2 = self.layernorm2(attn2 + out1)

        # 3. Feed Forward Network
        ffn_output = self.ffn(out2)
        out3 = self.layernorm3(ffn_output + out2)

        return out3

[ ]
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, num_heads, dff, input_vocab_size,
                 target_vocab_size, rate=0.1):
        super().__init__()
        self.encoder = nn.ModuleList([EncoderLayer(d_model, num_heads, dff, rate) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, num_heads, dff, rate) for _ in range(n_layers)])
        self.final_layer = nn.Linear(d_model, target_vocab_size)

    def forward(self, inp, tar, enc_mask, look_ahead_mask, dec_mask):
        enc_output = inp
        for layer in self.encoder:
            enc_output = layer(enc_output, enc_mask)

        dec_output = tar
        for layer in self.decoder:
            dec_output = layer(dec_output, enc_output, look_ahead_mask, dec_mask)

        return self.final_layer(dec_output)

[ ]
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)
        self.dropout3 = nn.Dropout(rate)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        # Self-attention
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(attn1 + x)

        # Encoder-Decoder attention
        attn2, attn_weights_block2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        out2 = self.layernorm2(attn2 + out1)

        # Feed forward
        ffn_output = self.ffn(out2)
        out3 = self.layernorm3(ffn_output + out2)

        return out3, attn_weights_block1, attn_weights_block2

[ ]
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, num_heads, dff, input_vocab_size,
                 target_vocab_size, pe_input, pe_target, rate=0.1):
        super().__init__()
        self.encoder = nn.ModuleList([EncoderLayer(d_model, num_heads, dff, rate) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, num_heads, dff, rate) for _ in range(n_layers)])
        self.final_layer = nn.Linear(d_model, target_vocab_size)

    def forward(self, inp, tar, enc_padding_mask, look_ahead_mask, dec_padding_mask):
        # Simplified forward pass focusing on the class structure
        enc_output = inp
        for layer in self.encoder:
            enc_output = layer(enc_output, enc_padding_mask)

        dec_output = tar
        for layer in self.decoder:
            dec_output, _, _ = layer(dec_output, enc_output, look_ahead_mask, dec_padding_mask)

        final_output = self.final_layer(dec_output)
        return final_output

[ ]
# 학습 가능 여부 체크
import torch

required_vars = ['train_loader', 'val_loader', 'transformer', 'optimizer', 'criterion', 'device']
missing = [v for v in required_vars if v not in globals()]

if not missing:
    print("✅ 모든 학습 준비가 완료되었습니다.")
    print(f"- 장치: {device}")
    print(f"- 학습 배치 수: {len(train_loader)}")

    # 1 에포크 테스트 실행
    print("\n--- 1 에포크 테스트 시작 ---")
    transformer.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        loss = train_step(transformer, batch, optimizer, criterion, device)
        total_loss += loss
        if (i + 1) % 100 == 0:
            print(f"Batch {i+1} Loss: {loss:.4f}")
        if i > 500: # 테스트용이므로 일부만 수행
            break

    print(f"\n테스트 완료! 평균 손실: {total_loss/(i+1):.4f}")
else:
    print(f"❌ 다음 변수가 누락되어 학습이 불가능합니다: {missing}")
    print("데이터 로더나 모델 정의 셀을 먼저 실행해 주세요.")
Colab 유료 제품 - 여기에서 계약 취소
v